# CIDS · 09 — S2c: the checker learns from the finder's real mistakes (hard negatives)

**Problem (S3b):** S2b rejects 97% of *synthetic* negative boxes but only **0.6% of SegResNet's real false candidates**, so the council cannot drop the false blobs.

**S2c** = S2b + training on SegResNet's **real** candidates:

| step | what | time (T4) |
|---|---|---|
| 1 · mine | SegResNet-100 on train / val / test → candidates → same SAM crop + box as the pipeline → frozen encoder → embedding. Each candidate is labelled **true** (touches tumour) or **false** (hard negative). Train: prob > 0.3, top 8 (more mistakes); val/test: 0.5, top 5 (= pipeline). | ~2–3 h |
| 2 · train | start from **S2b**, 30 epochs. Batches: 25% real false candidates → **empty mask**, 10% real true candidates → tumour, rest = the S2b mix (clean / messy / synthetic-negative boxes + clicks). Best epoch = mean(box1, messy1, **hard_reject**, **real_keep**) on val. | ~0.5–1 h |
| 3 · S3c | the full automatic council (same as notebook 06) with SegResNet-100 + **S2c** → test Dice | ~1.5–2 h |

**Watch:** `hard_reject` (val/test real false candidates rejected; S2b ≈ 0.01) should go **up**; `real_keep` (true candidates kept) must stay **≈ 1**; `box1` should stay ≈ 0.70. Epoch 0 = S2b, for reference.

**Settings:** GPU T4 x2 · **Internet ON** · Persistence: Files.
**Inputs:** cache · `cids-sam-emb` · `cids-checkpoints` (`segresnet_img_e100_best.pt`, `sam_s2b_best.pt`).
**After the run:** download `s2c/best.pt` → upload to `cids-checkpoints` as **`sam_s2c_best.pt`**. Optionally save `s2c_cands/` as a dataset (then set `CANDS_DIR` and skip mining next time).

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
EMB_ROOT   = '/kaggle/input/datasets/kumarvemuri1/cids-sam-emb/SAM_embeddings_dataset'
CKPT_DIR   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints'
SEG_CKPT   = f'{CKPT_DIR}/segresnet_img_e100_best.pt'
S2B_CKPT   = f'{CKPT_DIR}/sam_s2b_best.pt'

CANDS_DIR  = ''               # '' = mine now into /kaggle/working/s2c_cands ; or a dataset path from a previous run
EPOCHS     = 30
LR         = 2e-4             # fine-tuning from S2b -> lower than S2b's 5e-4
P_HARD     = 0.25             # share of batches = real false candidates (target: empty)
P_REALPOS  = 0.10             # share of batches = real true candidates (target: tumour)
RUN_S3C    = True             # run the full council with S2c at the end (step 3)

CODE_DIR = '/kaggle/working/cids'
OUT      = '/kaggle/working/s2c'
S3C_OUT  = '/kaggle/working/s3c_seg100_s2c'
MINE     = not CANDS_DIR
CANDS    = CANDS_DIR or '/kaggle/working/s2c_cands'
PATHS = [CACHE_ROOT, EMB_ROOT, SEG_CKPT, S2B_CKPT] + ([CANDS_DIR] if CANDS_DIR else [])

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q monai medim einops appdirs
!pytest -q tests

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
p = torch.cuda.get_device_properties(0); print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS), 'fix the paths above (Add Input -> the dataset; copy its path)'

## Smoke test (~5 min): mine 3 cases per split, train 1 epoch on 16 cases
Check: the mining prints `N candidates = x true + y FALSE`; training prints `epoch 0 (start)`, `batches: {'gt': .., 'hard': .., 'realpos': ..}` and a `REAL hard_reject … real_keep …` line; the test prints the **REAL candidates** table for `finetuned` and `compare` (= S2b). Numbers after 1 short epoch mean nothing.

In [ ]:
!python scripts/s2c_mine_candidates.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --out /kaggle/working/s2c_smoke_cands --limit 3 --only train,val,test
!python scripts/s2a_sam_finetune.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --box_aug realistic \
    --cand_root /kaggle/working/s2c_smoke_cands --init_ckpt {S2B_CKPT} --compare_ckpt {S2B_CKPT} --no_zero_shot \
    --out /kaggle/working/s2c_smoke --epochs 1 --val_every 1 --limit_train 16 --limit_eval 3
!rm -rf /kaggle/working/s2c_smoke /kaggle/working/s2c_smoke_cands

## Step 1 · mine the real candidates (~2–3 h; skipped if `CANDS_DIR` is set)
Resumable: if the session stops, run the cell again (finished cases are skipped). Look at the **number of FALSE candidates in train**: that is the hard-negative training set. If it is very small (< ~300), lower `--thr_train` to 0.2.

In [ ]:
if MINE:
    !python scripts/s2c_mine_candidates.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --out {CANDS}
import pandas as pd
for s in ['train', 'val', 'test']:
    f = Path(CANDS, f'{s}_index.csv')
    if f.exists():
        d = pd.read_csv(f)
        print(f"{s:5s}: {len(d):5d} candidates | true {int(d.is_true.sum()):5d} | FALSE {int((~d.is_true).sum()):5d} "
              f"| median ml true {d[d.is_true].seg_ml.median():.2f} / false {d[~d.is_true].seg_ml.median():.2f}")
!du -sh {CANDS}

## Step 2 · train S2c from S2b (~0.5–1 h)

In [ ]:
!python scripts/s2a_sam_finetune.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --box_aug realistic \
    --cand_root {CANDS} --p_hard {P_HARD} --p_realpos {P_REALPOS} --init_ckpt {S2B_CKPT} --compare_ckpt {S2B_CKPT} \
    --lr {LR} --warmup 1 --epochs {EPOCHS} --no_zero_shot --out {OUT}

In [ ]:
import json, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUT}/train_log.csv')
v = log.dropna(subset=['val_score'])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(log.epoch, log.loss); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
for c in ['val_box1', 'val_messy1', 'val_hard_reject', 'val_real_keep', 'val_neg_reject']:
    if c in v: ax[1].plot(v.epoch, v[c], marker='o', label=c[4:])
ax[1].legend(); ax[1].set_title('val (epoch 0 = S2b)'); plt.tight_layout(); plt.show()
s = json.load(open(f'{OUT}/test_summary.json'))
print('REAL candidates on test (finetuned = S2c, compare = S2b):')
print(pd.DataFrame(s['real_candidates']).T)
print(json.dumps({k: v for k, v in s.get('full_res_1mm', {}).items() if 'box1' in k or 'messy1' in k}, indent=1))

## Step 3 · S3c — full automatic council with S2c (~1.5–2 h)
Same script and rules as notebook 06 (S3b), only the checker changes, so the result is directly comparable with **S3b: segjury_0.7 = 0.699**. The new rules **`segkeep` / `samkeep`** now actually reject candidates.

In [ ]:
if RUN_S3C:
    !python scripts/s3a_seg2sam.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_lora_ckpt {OUT}/best.pt --out {S3C_OUT}
    r = json.load(open(f'{S3C_OUT}/summary.json'))
    print('chosen on val:', r['chosen_rule_on_val'], '| SAM rejection on test:', r['sam_rejection'])
    t = pd.DataFrame(r['test']).T
    print(t[['dice', 'hd95_median', 'nsd', 'missed']].sort_values('dice', ascending=False).round(4))
!du -sh /kaggle/working/*